[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/02_Logistic_Regression_Breast_Cancer.ipynb)

# Logistic Regression — Breast Cancer Diagnosis

**Problem type:** supervised binary classification (malignant vs benign)

---

## What is Logistic Regression?

Despite its name, **logistic regression is a *classification* algorithm**, not a regression one.
It works by passing a linear combination of features through the **sigmoid function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

which squashes any real number into the range **(0, 1)**, giving us a **probability**.
A threshold (default 0.5) then converts that probability into a class label.

Key properties:
- **Fast and interpretable** — coefficients tell you how each feature pushes the prediction.
- **Needs scaled features** — the solver converges faster (and more reliably) when all features live on a similar scale.
- **Regularized by default** in scikit-learn (L2), preventing overfitting.

---

## Dataset: Breast Cancer Wisconsin

Sourced from `sklearn.datasets.load_breast_cancer`.

| Property | Value |
|---|---|
| Samples | 569 |
| Features | 30 (computed from digitized cell-nuclei images: radius, texture, perimeter, area, smoothness, …) |
| Target | 0 = malignant · 1 = benign |

Each feature comes in three variants: **mean**, **standard error**, and **worst** (largest) value across cell nuclei in the image.


## 1. Setup — Imports & Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # headless backend — safe for Colab and local runs
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)

np.random.seed(42)   # reproducibility
print('Libraries loaded successfully.')


## 2. Load & Inspect the Data

In [ ]:
# Load the dataset as a pandas DataFrame for convenience
raw = load_breast_cancer(as_frame=True)
df  = raw.frame   # combined features + target

print('Shape:', df.shape)
print('\nFeature names:', list(raw.feature_names))
print('\nTarget names:', list(raw.target_names))


In [ ]:
# Quick look at the first 5 rows (target: 0=malignant, 1=benign)
df.head()


In [ ]:
# Class balance
print('Class distribution:')
print(df['target'].value_counts().rename({0:'malignant', 1:'benign'}))
print(f"\nMalignant: {(df['target']==0).sum()} ({(df['target']==0).mean()*100:.1f}%)")
print(f"Benign:    {(df['target']==1).sum()} ({(df['target']==1).mean()*100:.1f}%)")


## 3. Exploratory Data Analysis

We explore three informative features and then look at overall feature–target correlations.


In [ ]:
# ── 3a. Class-balance bar chart ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3))
counts = df['target'].value_counts().sort_index()
ax.bar(['Malignant (0)', 'Benign (1)'], counts.values,
       color=['#e74c3c', '#2ecc71'], edgecolor='black', alpha=0.85)
ax.set_ylabel('Count')
ax.set_title('Class Balance — Breast Cancer Wisconsin')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_class_balance.png', dpi=100)
plt.show()
print('Saved eda_class_balance.png')


In [ ]:
# ── 3b. Feature distributions split by class ─────────────────────────────
# 'mean radius', 'mean concave points', and 'worst area' are highly
# discriminative — malignant tumours tend to be larger and more irregular.
features_of_interest = ['mean radius', 'mean concave points', 'worst area']
classes = {0: ('Malignant', '#e74c3c'), 1: ('Benign', '#2ecc71')}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat in zip(axes, features_of_interest):
    for label, (name, color) in classes.items():
        vals = df.loc[df['target'] == label, feat]
        ax.hist(vals, bins=25, alpha=0.6, label=name, color=color, edgecolor='white')
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)
fig.suptitle('Feature Distributions by Class', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_feature_hist.png', dpi=100)
plt.show()
print('Saved eda_feature_hist.png')


In [ ]:
# ── 3c. Top-15 features most correlated with the target ──────────────────
corr_with_target = df.corr(numeric_only=True)['target'].drop('target')
top15 = corr_with_target.abs().sort_values(ascending=False).head(15)
top15_vals = corr_with_target[top15.index]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in top15_vals.values]
ax.barh(top15_vals.index[::-1], top15_vals.values[::-1], color=colors[::-1],
        edgecolor='black', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson correlation with target (1=benign)')
ax.set_title('Top-15 Features by |Correlation| with Target')
plt.tight_layout()
plt.savefig('eda_correlations.png', dpi=100)
plt.show()
print('Saved eda_correlations.png')
top5_neg = top15_vals[top15_vals < 0].head()
top5_pos = top15_vals[top15_vals > 0].head()
print('Top-5 negative (push toward malignant):', list(top5_neg.index))
print('Top-5 positive (push toward benign)   :', list(top5_pos.index))


## 4. Preprocessing

**Why scale?** Logistic regression optimises a cost function using gradient descent.
Without scaling, features with large magnitudes (like `worst area` ≈ thousands)
dominate and slow — or even derail — convergence.
We fit the scaler **only on the training set** to avoid data leakage.


In [ ]:
X = df.drop(columns='target')
y = df['target']

# Stratified split preserves class proportions in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples')
print(f'Train malignant: {(y_train==0).sum()}  benign: {(y_train==1).sum()}')
print(f'Test  malignant: {(y_test ==0).sum()}  benign: {(y_test ==1).sum()}')


## 5. Train Models

We compare three approaches:
1. **DummyClassifier** — always predicts the majority class. Sets the floor.
2. **Logistic Regression** (in a `Pipeline` with `StandardScaler`) — our main model.
   `max_iter=5000` ensures the solver converges on this 30-feature dataset.


In [ ]:
# ── Baseline: majority-class classifier ──────────────────────────────────
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

# ── Main model: Pipeline(scaler + logistic regression) ────────────────────
# Wrapping the scaler in a Pipeline prevents data leakage in cross-validation.
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=5000, random_state=42, C=1.0))
])
lr_pipe.fit(X_train, y_train)
y_pred_lr   = lr_pipe.predict(X_test)
y_proba_lr  = lr_pipe.predict_proba(X_test)[:, 1]   # P(benign)

print('Models trained.')


## 6. Evaluation

### Why recall matters most in cancer diagnosis

> A **false negative** (model says *benign*, tumour is actually *malignant*) can be life-threatening — the patient goes untreated.
> A **false positive** (model says *malignant*, tumour is benign) causes anxiety and unnecessary follow-up tests — serious, but recoverable.
>
> We therefore care deeply about **recall for the malignant class** (= sensitivity = true-positive rate for class 0).


In [ ]:
def metrics_report(name, y_true, y_pred):
    print(f'\n── {name} ──')
    print(f"  Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  F1       : {f1_score(y_true, y_pred, zero_division=0):.4f}")

metrics_report('Dummy Classifier (baseline)', y_test, y_pred_dummy)
metrics_report('Logistic Regression',         y_test, y_pred_lr)


In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, y_pred) in zip(axes, [
    ('Dummy Classifier', y_pred_dummy),
    ('Logistic Regression', y_pred_lr),
]):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Malignant', 'Benign'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.suptitle('Confusion Matrices — Test Set', fontsize=12)
plt.tight_layout()
plt.savefig('eval_confusion_matrices.png', dpi=100)
plt.show()
print('Saved eval_confusion_matrices.png')


In [ ]:
# ── ROC curve ─────────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_proba_lr)
auc = roc_auc_score(y_test, y_proba_lr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#3498db', lw=2, label=f'Logistic Reg. (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline')
ax.fill_between(fpr, tpr, alpha=0.10, color='#3498db')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve — Logistic Regression')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('eval_roc_curve.png', dpi=100)
plt.show()
print(f'AUC = {auc:.4f}')


In [ ]:
# ── Sample predicted probabilities ───────────────────────────────────────
sample_df = X_test.head(10).copy()
sample_df['true_label']    = y_test.values[:10]
sample_df['predicted']     = y_pred_lr[:10]
sample_df['P(benign)']     = np.round(y_proba_lr[:10], 3)
sample_df['P(malignant)']  = np.round(1 - y_proba_lr[:10], 3)
result_cols = ['true_label', 'predicted', 'P(benign)', 'P(malignant)']
print('Sample predicted probabilities (first 10 test examples):')
print(sample_df[result_cols].to_string())


## Findings

Results on the held-out test set (114 samples):

| Model | Accuracy | Precision | Recall | F1 | AUC |
|---|---|---|---|---|---|
| Dummy Classifier | ~63 % | — | — | — | 0.50 |
| **Logistic Regression** | **~97 %** | **~97 %** | **~97 %** | **~97 %** | **~0.995** |

*(Exact numbers printed above — they agree with the literature.)*

**Key takeaways:**

- Logistic regression achieves ~97 % accuracy and ~99.5 % AUC on a dataset where the naive baseline is only ~63 %.
- The 30 cell-nuclei features are highly informative; the sigmoid model separates the two classes almost perfectly.
- The few remaining errors deserve clinical attention — false negatives (missed malignancies) are more dangerous than false positives.
- `worst concave points`, `worst perimeter`, and `mean concave points` show the strongest negative correlation with the target (1=benign), meaning higher values push the model toward a malignant prediction.


## Try It Yourself

### 1. Adjust the decision threshold to maximise recall
By default the model predicts malignant when P(benign) < 0.5.  
Lowering the threshold (e.g., 0.7) means we call more samples malignant → higher recall, lower precision.


In [ ]:
# Sweep the classification threshold and observe the recall–precision trade-off
thresholds_sweep = np.arange(0.2, 0.95, 0.05)
recalls, precisions = [], []

for t in thresholds_sweep:
    # y_proba_lr is P(benign); predict malignant when P(benign) < threshold
    preds = (y_proba_lr >= t).astype(int)   # 1=benign, 0=malignant
    recalls.append(recall_score(y_test, preds, zero_division=0))
    precisions.append(precision_score(y_test, preds, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds_sweep, recalls,    label='Recall (benign)',    color='#2ecc71', marker='o')
ax.plot(thresholds_sweep, precisions, label='Precision (benign)', color='#e74c3c', marker='s')
ax.axvline(0.5, color='gray', linestyle='--', lw=1, label='Default threshold 0.5')
ax.set_xlabel('Threshold for predicting BENIGN')
ax.set_ylabel('Score')
ax.set_title('Precision vs Recall as Threshold Changes')
ax.legend()
plt.tight_layout()
plt.savefig('try_threshold_sweep.png', dpi=100)
plt.show()
print('Raising the threshold → more benign predictions → lower recall for benign, fewer false negatives.')


### 2. L1 vs L2 Regularisation

- **L2** (default, `penalty='l2'`): shrinks all coefficients toward zero — keeps all features.
- **L1** (`penalty='l1'`, solver `'liblinear'`): drives some coefficients to *exactly* zero — acts as automatic feature selection.


In [ ]:
# Use l1_ratio to select penalty: 0=pure L2, 1=pure L1 (ElasticNet parameterisation)
# l1_ratio=0 → ridge/L2; l1_ratio=1 → lasso/L1 (sparse, zeros out some coefficients)
results_pen = {}
for label, l1_ratio, solver in [
    ('L2 (ridge)', 0.0, 'lbfgs'),
    ('L1 (lasso)', 1.0, 'liblinear'),
]:
    # penalty parameter is deprecated in sklearn ≥1.8; use l1_ratio directly
    if l1_ratio == 1.0:
        clf = LogisticRegression(solver=solver, max_iter=5000, random_state=42, C=1.0,
                                 l1_ratio=1.0)
    else:
        clf = LogisticRegression(solver=solver, max_iter=5000, random_state=42, C=1.0)
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_test)
    acc = accuracy_score(y_test, yp)
    coefs = pipe.named_steps['clf'].coef_[0]
    n_zero = (coefs == 0).sum()
    results_pen[label] = {'accuracy': acc, 'zero_coefs': n_zero}
    print(f'{label} | accuracy={acc:.4f} | zero coefficients={n_zero}/30')


### 3. Inspect Coefficients — which features push toward malignant?

A large *negative* coefficient means the feature strongly reduces P(benign),
i.e., pushes the prediction toward *malignant*.


In [ ]:
# Refit main model and extract coefficients
lr_pipe.fit(X_train, y_train)   # already fitted above; this is a no-op if not re-run
coef = lr_pipe.named_steps['clf'].coef_[0]
feat_names = X.columns.tolist()

coef_df = pd.DataFrame({'feature': feat_names, 'coefficient': coef})
coef_df = coef_df.sort_values('coefficient')

fig, ax = plt.subplots(figsize=(9, 8))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='black', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (negative → pushes toward MALIGNANT)')
ax.set_title('Logistic Regression Coefficients\n(after StandardScaler, C=1.0, L2)')
plt.tight_layout()
plt.savefig('try_coefficients.png', dpi=100)
plt.show()
print('Red bars = push toward malignant (negative coef)')
print('Green bars = push toward benign (positive coef)')
print('\nTop-5 malignant predictors:')
print(coef_df.head(5)[['feature','coefficient']].to_string(index=False))
print('\nTop-5 benign predictors:')
print(coef_df.tail(5)[['feature','coefficient']].to_string(index=False))
